# Day 6: Dictionaries, tuples, and relative permeability

**Time:** about 1 hour  |  **Needs:** Days 1 to 5
**Reservoir idea:** relative permeability: how easily each fluid flows when both are in the pores (the Corey curves)
**Python you will learn:** dictionaries (lookup by name, counting, totals per key), tuples, sorting, a dictionary of dictionaries

By the end of today you will be able to:

1. store named properties in a dictionary and look them up by name,
2. use a dictionary to total, count or find the peak **per well**, however many wells there are,
3. use tuples for pairs and triples, return several values, and sort a list of tuples,
4. calculate the Corey relative-permeability curves of oil and water, and compare two rock types,
5. find the saturation where the two curves cross.

> **Companion reading (optional, about 50 minutes).** Charles Severance, *Python for Everybody* (free online: <https://www.py4e.com/html3/>):
> - Chapter 9 "Dictionaries": "Dictionary as a set of counters", "Dictionaries and files", "Looping and dictionaries", "Advanced text parsing"
> - Chapter 10 "Tuples": "Tuples are immutable", "Comparing tuples", "Tuple assignment", "Dictionaries and tuples", "Multiple assignment with dictionaries", "The most common words", "Using tuples as keys in dictionaries"

## 0. Setup and recap

Run the next cell. It recreates the Day 5 production log and the `read_log` function (given here so that today stands alone).

**Recap of Day 5.** Files are read line by line; `.split()` cuts a line into words; a table is a list of rows; `read_log` returns a list of `[date, well, oil, water]` records and a count of bad lines.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- recreate the Day 5 production log -------------------------------------
lines = ["# date well oil_rate(STB/d) water_rate(STB/d)"]
for d in range(30):
    date = f"2024-01-{d + 1:02d}"
    p1_oil = round(2500 * 0.985 ** d)
    p2_oil = round(1800 * 0.99 ** d)
    lines.append(f"{date} P1 n/a {10 * d}" if d == 7 else f"{date} P1 {p1_oil} {10 * d}")
    lines.append(f"{date} P2 {p2_oil}" if d == 15 else f"{date} P2 {p2_oil} {3 * d}")
with open("production_log.txt", "w") as f:
    f.write("\n".join(lines) + "\n")


def read_log(filename):
    """Read the production log. Returns (records, n_bad)."""
    records = []
    n_bad = 0
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("#"):
                continue
            words = line.split()
            try:
                oil = float(words[2])
                water = float(words[3])
            except (ValueError, IndexError):
                n_bad += 1
                continue
            records.append([words[0], words[1], oil, water])
    return records, n_bad


records, n_bad = read_log("production_log.txt")
print(len(records), "good records,", n_bad, "bad lines")

## 1. Story: one rule, two hundred wells

*(An invented but typical situation.)* The production report of a field has to show, for each well, the total oil, the peak rate and the number of days on production. On Day 5 you wrote `if well == "P1": ... elif well == "P2": ...`. That works for two wells. For 200 wells you would write 200 branches, and a new well would need a new branch.

What you want is a structure that says *"this well's total"* by the **name** of the well. Python has exactly that: the **dictionary**.

## 2. Dictionaries

A list finds items by **position** (`values[2]`). A dictionary finds items by a **key**: a name, a number, or other fixed value. Each entry is a **key: value** pair.

```python
rock = {"phi": 0.20, "k_md": 150.0, "swc": 0.2}
rock["phi"]            # 0.2
```

Run this and predict each printed line first.

In [ ]:
rock = {"phi": 0.20, "k_md": 150.0, "swc": 0.2}

print(rock["phi"])             # look up by key
rock["sor"] = 0.25             # add a new entry
rock["phi"] = 0.18             # change an existing entry
print(rock)
print(len(rock))               # number of entries
print("k_md" in rock)          # is this a key?
print("kv_md" in rock)

Things to know:

- Keys are usually text (or numbers) and must be **unique**; assigning to an existing key replaces the value.
- Asking for a key that does not exist is an error (`KeyError`). `d.get(key, default)` returns a default instead:

In [ ]:
print(rock.get("phi", 0.1))        # key exists: its value
print(rock.get("kv_md", 15.0))     # key is missing: the default
# print(rock["kv_md"])             # remove the # to see the KeyError

### Looping over a dictionary

| You write | You get |
|---|---|
| `for key in d:` | each key |
| `for value in d.values():` | each value |
| `for key, value in d.items():` | each key together with its value |

Dictionaries keep the order in which entries were added.

In [ ]:
for name, value in rock.items():
    print(f"{name:>5} = {value}")

**Your turn 2.1.** Create a dictionary `corey` with the parameters of a pair of Corey relative-permeability curves, with the keys and values:

| key | value |
|---|---|
| `"swc"` | 0.2 (connate water saturation) |
| `"sor"` | 0.2 (residual oil saturation) |
| `"krw_max"` | 0.4 (water relative permeability at the end point) |
| `"kro_max"` | 1.0 (oil relative permeability at the end point) |
| `"nw"` | 2.0 (water exponent) |
| `"no"` | 2.0 (oil exponent) |

In [ ]:
# YOUR CODE HERE
corey = None

In [ ]:
# Check (do not edit)
assert corey == {"swc": 0.2, "sor": 0.2, "krw_max": 0.4, "kro_max": 1.0, "nw": 2.0, "no": 2.0}, f"corey is {corey}"
print("Correct")

<details><summary>Solution</summary>

```python
corey = {"swc": 0.2, "sor": 0.2, "krw_max": 0.4, "kro_max": 1.0, "nw": 2.0, "no": 2.0}
```
</details>

**Your turn 2.2.** A settings dictionary may be missing entries. With `settings = {"dt": 0.05}`, use `.get` to read `dt_value` (the dictionary has it) and `t_end_value` (it does not; use the default 2.0).

In [ ]:
settings = {"dt": 0.05}

# YOUR CODE HERE
dt_value = None
t_end_value = None

In [ ]:
# Check (do not edit)
assert dt_value == 0.05, f"dt_value should be 0.05, got {dt_value}"
assert t_end_value == 2.0, f"t_end_value should be 2.0 (the default), got {t_end_value}"
print("Correct")

<details><summary>Solution</summary>

```python
dt_value = settings.get("dt", 1.0)
t_end_value = settings.get("t_end", 2.0)
```
</details>

## 3. A dictionary as a set of counters: totals per well

The pattern you need: *for each record, find the entry of its well and add to it*. The first time a well is seen it has no entry yet, so start from 0. `.get(well, 0)` does exactly that:

```python
totals = {}
for record in records:
    well = record[1]
    oil = record[2]
    totals[well] = totals.get(well, 0) + oil
```
Read it as: *new total for this well = (old total, or 0 if none yet) + this day's oil*. Run it:

In [ ]:
totals = {}
for record in records:
    well = record[1]
    oil = record[2]
    totals[well] = totals.get(well, 0) + oil

print(totals)

No `if`/`elif` per well, and it works for 2 wells or 200. The same pattern **counts**: replace `+ oil` by `+ 1`.

**Your turn 3.1.** Write `totals_by_well(records)` that returns a dictionary `{well: total oil}`.

In [ ]:
def totals_by_well(records):
    """Dictionary with the total oil of each well."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
t = totals_by_well(records)
assert set(t.keys()) == {"P1", "P2"}, f"the keys should be the well names, got {list(t.keys())}"
assert t["P1"] == 58506 and t["P2"] == 45307, f"totals are wrong: {t}"
assert totals_by_well([]) == {}, "an empty list of records should give an empty dictionary"
print("Correct:", t)

<details><summary>Solution</summary>

```python
def totals_by_well(records):
    """Dictionary with the total oil of each well."""
    totals = {}
    for record in records:
        well = record[1]
        oil = record[2]
        totals[well] = totals.get(well, 0) + oil
    return totals
```
</details>

**Your turn 3.2.** The **peak** pattern per key: write `peak_by_well(records)` returning `{well: highest oil rate}`. For a well that is not in the dictionary yet, the first rate is its peak so far; otherwise compare. (Test with `well not in peaks`.)

In [ ]:
def peak_by_well(records):
    """Dictionary with the highest oil rate of each well."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
peaks = peak_by_well(records)
assert peaks == {"P1": 2500.0, "P2": 1800.0}, f"peaks are wrong: {peaks}"
assert peak_by_well([["d", "A", 5.0, 0.0], ["d", "A", 9.0, 0.0], ["d", "A", 7.0, 0.0]]) == {"A": 9.0}, "the highest of 5, 9, 7 is 9"
print("Correct:", peaks)

<details><summary>Solution</summary>

```python
def peak_by_well(records):
    """Dictionary with the highest oil rate of each well."""
    peaks = {}
    for record in records:
        well = record[1]
        oil = record[2]
        if well not in peaks or oil > peaks[well]:
            peaks[well] = oil
    return peaks
```
(`or` stops at the first true part: if `well not in peaks` is true, `peaks[well]` is never evaluated, so there is no `KeyError`.)
</details>

**Your turn 3.3.** Write `wells_above(totals, limit)`: the list of well names whose total is above `limit`, sorted alphabetically. Loop with `.items()`.

In [ ]:
def wells_above(totals, limit):
    """Sorted list of the wells whose total is above limit."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert wells_above({"P2": 60, "P1": 70, "P3": 10}, 50) == ["P1", "P2"], "wells above 50, in alphabetical order"
assert wells_above(t, 50000) == ["P1"], f"only P1 is above 50000, got {wells_above(t, 50000)}"
assert wells_above({}, 0) == [], "no wells: empty list"
print("Correct")

<details><summary>Solution</summary>

```python
def wells_above(totals, limit):
    """Sorted list of the wells whose total is above limit."""
    names = []
    for well, total in totals.items():
        if total > limit:
            names.append(well)
    names.sort()
    return names
```
</details>

## 4. Tuples

A **tuple** is like a list that cannot be changed after it is made. It is written with round brackets.

```python
point = (2000, 1.5)       # a pressure and a Bo
point[0]                  # 2000, indexing works as for lists
point[0] = 2500           # TypeError: tuples are immutable
```

You already used tuples without the name: on Day 2, `return times, values` returned a tuple and `times, values = euler_growth(...)` **unpacked** it. Unpacking works anywhere:

In [ ]:
point = (2000, 1.5)
pressure, bo = point            # unpacking: one name for each item
print(pressure, bo)

a, b = 1, 2
a, b = b, a                     # swap two values in one line
print(a, b)

for well, total in totals.items():   # .items() gives (key, value) tuples
    print(well, total)

### Tuples compare item by item, so lists of tuples can be sorted

When Python compares two tuples, it compares the first items; only if they tie does it look at the second, and so on. So `.sort()` on a list of tuples sorts by the first item. This is the standard trick to **rank a dictionary by its values**: swap each pair into `(value, key)`, sort, and reverse.

In [ ]:
pairs = [(45307.0, "P2"), (58506.0, "P1"), (12000.0, "P3")]
pairs.sort(reverse=True)        # biggest first
print(pairs)
for total, well in pairs:
    print(f"{well}: {total:,.0f} STB")

**Your turn 4.1.** Write `ranking(totals)`, which takes a dictionary `{well: total}` and returns a list of `(total, well)` tuples, **largest total first**.

In [ ]:
def ranking(totals):
    """List of (total, well) tuples, largest total first."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert ranking({"A": 5, "B": 9, "C": 7}) == [(9, "B"), (7, "C"), (5, "A")], f"got {ranking({'A': 5, 'B': 9, 'C': 7})}"
assert ranking(t) == [(58506.0, "P1"), (45307.0, "P2")], f"got {ranking(t)}"
print("Correct")

<details><summary>Solution</summary>

```python
def ranking(totals):
    """List of (total, well) tuples, largest total first."""
    pairs = []
    for well, total in totals.items():
        pairs.append((total, well))
    pairs.sort(reverse=True)
    return pairs
```
</details>

### Tuples as dictionary keys

A list cannot be a dictionary key (it can change), but a tuple can. Use it when the key has two parts, such as *(well, day)*:

In [ ]:
oil_by_well_day = {}
for record in records:
    date, well, oil = record[0], record[1], record[2]
    oil_by_well_day[(well, date)] = oil

print(oil_by_well_day[("P2", "2024-01-03")])

This is how a simulator can name a grid block: the key `(i, j, k)` is a tuple. (It will return on Day 10.)

### A report

Putting it together, the ranking, with the formatting of Day 4:

In [ ]:
print(f"{'rank':>4} {'well':>5} {'oil (STB)':>11}")
rank = 1
for total, well in ranking(totals_by_well(records)):
    print(f"{rank:>4} {well:>5} {total:>11,.0f}")
    rank += 1

## 5. Relative permeability

### The physics

When oil and water share the pores, each flows as if the other took up part of the rock. The **relative permeability** $k_r$ of a fluid (a number between 0 and 1) says what fraction of the rock's permeability that fluid can use at a given saturation. It depends on the water saturation $S_w$:

- water cannot flow until $S_w$ exceeds the **connate water saturation** $S_{wc}$ (the water stuck to the grains), so $k_{rw}=0$ at $S_w\le S_{wc}$,
- oil cannot flow when the oil saturation falls to the **residual oil saturation** $S_{or}$, so $k_{ro}=0$ at $S_w \ge 1-S_{or}$.

A common model is the **Corey** form. First define a normalised saturation, which runs from 0 at $S_{wc}$ to 1 at $1-S_{or}$:

$$s = \frac{S_w - S_{wc}}{1 - S_{wc} - S_{or}} \tag{1}$$

then

$$k_{rw} = k_{rw}^{\max}\; s^{\,n_w}, \qquad k_{ro} = k_{ro}^{\max}\,(1-s)^{\,n_o} \tag{2}$$

with end points $k_{rw}^{\max}$ and $k_{ro}^{\max}$ and exponents $n_w$, $n_o$. Outside $S_{wc} \le S_w \le 1-S_{or}$ we hold $s$ at 0 or 1 (that gives the flat ends).

The numbers in your dictionary `corey` from Section 2 are exactly these parameters.

### By hand first

With `corey`: $S_w = 0.5$ gives $s = (0.5-0.2)/(1-0.2-0.2) = 0.5$. Then $k_{rw}= 0.4\times 0.5^2 = 0.1$ and $k_{ro} = 1.0 \times (1-0.5)^2 = 0.25$.

**Your turn 5.1.** Write `corey_krw(sw, props)` and `corey_kro(sw, props)`, where `props` is a dictionary like `corey`. Compute the normalised saturation `s` from equation (1), clamp it to the range 0 to 1 (`s = min(max(s, 0.0), 1.0)`), and return the value from equation (2).

In [ ]:
def corey_krw(sw, props):
    """Water relative permeability from the Corey equations (1) and (2)."""
    # YOUR CODE HERE
    pass


def corey_kro(sw, props):
    """Oil relative permeability from the Corey equations (1) and (2)."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert abs(corey_krw(0.5, corey) - 0.1) < 1e-12, f"krw(0.5) should be 0.1, got {corey_krw(0.5, corey)}"
assert abs(corey_kro(0.5, corey) - 0.25) < 1e-12, f"kro(0.5) should be 0.25, got {corey_kro(0.5, corey)}"
assert corey_krw(0.2, corey) == 0 and corey_kro(0.2, corey) == 1.0, "at Swc: krw = 0 and kro = kro_max"
assert abs(corey_krw(0.8, corey) - 0.4) < 1e-12 and corey_kro(0.8, corey) == 0, "at 1 - Sor: krw = krw_max and kro = 0"
assert corey_krw(0.1, corey) == 0 and corey_kro(0.9, corey) == 0, "outside the range the curves stay flat (clamp s)"
print("Correct")

<details><summary>Solution</summary>

```python
def corey_krw(sw, props):
    """Water relative permeability from the Corey equations (1) and (2)."""
    s = (sw - props["swc"]) / (1 - props["swc"] - props["sor"])
    s = min(max(s, 0.0), 1.0)
    return props["krw_max"] * s ** props["nw"]


def corey_kro(sw, props):
    """Oil relative permeability from the Corey equations (1) and (2)."""
    s = (sw - props["swc"]) / (1 - props["swc"] - props["sor"])
    s = min(max(s, 0.0), 1.0)
    return props["kro_max"] * (1 - s) ** props["no"]
```
</details>

**Your turn 5.2.** Write `rel_perm_table(props, n)` that returns a list of `n` tuples `(sw, krw, kro)`, with `sw` evenly spaced from `swc` to `1 - sor` (both ends included). The $i$-th point (starting at $i = 0$) is at $S_{wc} + (1-S_{wc}-S_{or})\,\dfrac{i}{n-1}$.

In [ ]:
def rel_perm_table(props, n):
    """List of n tuples (sw, krw, kro) from swc to 1 - sor."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
tab = rel_perm_table(corey, 7)
assert len(tab) == 7, f"expected 7 rows, got {len(tab)}"
assert abs(tab[0][0] - 0.2) < 1e-12 and abs(tab[0][1]) < 1e-12 and abs(tab[0][2] - 1.0) < 1e-12, f"first row should be (0.2, 0, 1), got {tab[0]}"
assert abs(tab[-1][0] - 0.8) < 1e-12 and abs(tab[-1][1] - 0.4) < 1e-12 and abs(tab[-1][2]) < 1e-12, f"last row should be (0.8, 0.4, 0), got {tab[-1]}"
assert abs(tab[3][0] - 0.5) < 1e-12 and abs(tab[3][1] - 0.1) < 1e-12, f"the middle row should be at Sw = 0.5 with krw = 0.1, got {tab[3]}"
print("Correct")
for sw, krw, kro in tab:
    print(f"  Sw = {sw:.2f}   krw = {krw:.3f}   kro = {kro:.3f}")

<details><summary>Solution</summary>

```python
def rel_perm_table(props, n):
    """List of n tuples (sw, krw, kro) from swc to 1 - sor."""
    rows = []
    width = 1 - props["swc"] - props["sor"]
    for i in range(n):
        sw = props["swc"] + width * i / (n - 1)
        rows.append((sw, corey_krw(sw, props), corey_kro(sw, props)))
    return rows
```
</details>

Now plot the two curves. The `for sw, krw, kro in table` loop **unpacks** each tuple into three names.

In [ ]:
table = rel_perm_table(corey, 61)

sw_list = []
krw_list = []
kro_list = []
for sw, krw, kro in table:
    sw_list.append(sw)
    krw_list.append(krw)
    kro_list.append(kro)

plt.plot(sw_list, krw_list, label="krw (water)")
plt.plot(sw_list, kro_list, label="kro (oil)")
plt.xlabel("water saturation Sw")
plt.ylabel("relative permeability")
plt.legend()
plt.grid(True)
plt.show()

Water gets easier to move as its saturation grows, oil harder. Both are poor in the middle: the two fluids hinder each other.

**Your turn 5.3.** The curves cross where $k_{rw}=k_{ro}$. Using a table with 201 points, find the saturation of the crossing: build a list of tuples `(abs(krw - kro), sw)`, one per row, and take the `min(...)` of the list. (Tuples compare item by item, so the minimum is the row where the difference is smallest.) Store the saturation in `crossover_sw`.

In [ ]:
table = rel_perm_table(corey, 201)

# YOUR CODE HERE
crossover_sw = None

In [ ]:
# Check (do not edit)
assert abs(crossover_sw - 0.5676) < 0.005, f"crossover_sw should be about 0.5676, got {crossover_sw}"
print(f"Correct: the curves cross at Sw = {crossover_sw:.3f}")

<details><summary>Solution</summary>

```python
candidates = []
for sw, krw, kro in table:
    candidates.append((abs(krw - kro), sw))
best = min(candidates)           # smallest difference
crossover_sw = best[1]
```
The exact answer solves $0.4\,s^2=(1-s)^2$: $s = 0.6126$, so $S_w = 0.2 + 0.6\times0.6126 = 0.568$.
</details>

The crossover is a quick way to describe how **wet** the rock is: in a water-wet rock the curves cross at a *high* water saturation (oil flows better for longer), in an oil-wet rock at a lower one.

### A dictionary of dictionaries

To compare two rocks, keep one parameter dictionary per rock **inside another dictionary**, keyed by the name of the rock. (The values are only illustrative, chosen to give clearly different curves.)

In [ ]:
rock_types = {
    "water-wet": {"swc": 0.20, "sor": 0.25, "krw_max": 0.30, "kro_max": 1.0, "nw": 2.5, "no": 2.0},
    "oil-wet":   {"swc": 0.10, "sor": 0.20, "krw_max": 0.70, "kro_max": 0.9, "nw": 1.5, "no": 3.0},
}

for rock_name, props in rock_types.items():
    table = rel_perm_table(props, 61)
    sw_list = [row[0] for row in table]
    plt.plot(sw_list, [row[1] for row in table], label=f"krw, {rock_name}")
    plt.plot(sw_list, [row[2] for row in table], "--", label=f"kro, {rock_name}")

plt.xlabel("water saturation Sw")
plt.ylabel("relative permeability")
plt.legend()
plt.grid(True)
plt.show()

(The two lines that build `sw_list` and the `plt.plot` arguments use a shortcut called a *list comprehension*, `[row[0] for row in table]`: it means "a list of `row[0]` for every `row` in `table`". It is the same as the loops you wrote above, in one line. You may use either.)

The two rocks give very different curves, and so a different oil recovery. That is why relative permeability is among the most important, and most uncertain, inputs of a simulation.

## 6. Bridge: the same equations inside `kusim`

Open `src/kusim/cases.py`. The function `corey_rock_fluid` builds relative-permeability tables from the same equations:

```python
def corey_rock_fluid(swc=0.2, sor=0.2, krw_max=0.4, kro_max=1.0, nw=2.0, no=2.0, ...):
```

Look at the parameter names and defaults: they are the **keys and values of your `corey` dictionary**. Its docstring states $k_{ro}=k_{ro}^{\max}(1-s)^{n_o}$, $k_{rw}=k_{rw}^{\max}s^{n_w}$ with the normalised saturation $s$, equations (1) and (2) above. The library computes all points at once with NumPy arrays (Day 7); you computed them one at a time in a loop.

The thesis case itself does not use a Corey formula: its relative permeabilities come as a **table** (the `SW KRO KRW PCOW` block of `legacy/fortran-3d3/INPUT.DAT`, 23 rows), which the simulator reads and interpolates with exactly the `interp` function you wrote on Day 5. (Its oil curve starts at $k_{ro} = 0.8$, not 1, and its table ends at $S_w=1$ with the water curve going to 1.) Tables like that are what real laboratory measurements look like; Corey curves are a convenient way to make a plausible table when you have none.

## 7. Self-quiz

Write short answers as comments, then open the answers.

1. `d = {"a": 1, "b": 2}`. What do `d["b"]`, `d.get("c", 0)` and `len(d)` give? What does `d["c"]` do?
2. What does `counts[word] = counts.get(word, 0) + 1` do, step by step?
3. When would you use a dictionary instead of a list? When a tuple instead of a list?
4. `pairs = [(3, "x"), (1, "y"), (2, "z")]`; what is `sorted(pairs)`?
5. With the `corey` parameters, what is $k_{ro}$ at $S_w=0.2$ and what is $k_{rw}$ at $S_w = 0.8$? Why?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. `2`, `0`, `2`. `d["c"]` raises a `KeyError`, because there is no key `"c"`.
2. It reads the current count for `word` (0 if the word has no entry yet), adds 1, and stores the result back under the same key.
3. Dictionary: when you look things up by a name or label (well name, property name). Tuple: for a small fixed group of values that belong together and should not change, such as a pair or a coordinate; it can also be a dictionary key.
4. `[(1, "y"), (2, "z"), (3, "x")]`: tuples compare by the first item.
5. $k_{ro}=1.0$ (all end-point mobility for oil, no water to hinder it) and $k_{rw} = 0.4$ (the water end point), because at $S_{wc}$ we have $s=0$ and at $1-S_{or}$ we have $s=1$.
</details>

## 8. Pocket reference (today's Python)

| You want to... | Write |
|---|---|
| make a dictionary | `d = {"phi": 0.2, "k": 150}` or `d = {}` |
| look up / set | `d["phi"]`, `d["phi"] = 0.18` |
| look up with a default | `d.get("sor", 0.2)` |
| is it a key? | `"phi" in d` |
| loop | `for key in d:`, `for key, value in d.items():`, `d.values()` |
| total per key | `d[k] = d.get(k, 0) + x` |
| remove | `del d[key]` or `d.pop(key)` |
| a tuple | `(a, b)`; unpack with `a, b = pair` |
| swap | `a, b = b, a` |
| sort tuples | `pairs.sort(reverse=True)` |
| smallest tuple | `min(list_of_tuples)` |
| tuple as a key | `d[(well, date)] = value` |
| list in one line | `[row[0] for row in table]` |

## 9. Wrap-up

Today you:

- stored named properties in dictionaries and looked them up by key,
- replaced the `if`/`elif` per well by `d[k] = d.get(k, 0) + x`, which works for any number of wells,
- used tuples for pairs and triples, unpacked them, and ranked wells by sorting tuples,
- coded the Corey relative-permeability curves from a parameter dictionary, found their crossover, and compared two rock types in a dictionary of dictionaries,
- saw the same parameters and equations in `kusim`'s `corey_rock_fluid`.

**Next, Day 7:** NumPy. Today you computed the curves one point at a time with a loop; with NumPy arrays one line will compute all points at once, and we add capillary pressure.